## Module 9: Optimizers, Learning Rate Strategies

In [4]:
# Std SGD with momentum
import torch.nn as nn
import torch.optim as optim
model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 1))
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum = 0.9, nesterov = False)

# Nesterov Momentum
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum = 0.9, nesterov = True, weight_decay = 1e-4)

# Adam
optimizer = optim.Adam(model.parameters(), lr = 0.001, betas = (0.9, 0.999), eps = 1e-8, weight_decay = 1e-4)

# AdamW
optimizer = optim.AdamW(model.parameters(), lr = 0.001, betas = (0.9, 0.999), eps = 1e-8, weight_decay = 1e-4)


In [7]:
# Learning Rate Range Test
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

def lr_range_test(model, train_loader, criteion, device, start_lr = 1e-7, end_lr = 1.0, 
                  n_steps = 100, smoothing = 0.05):
    lr_multiplier = (end_lr / start_lr) ** (1 /n_steps)
    opitimizer = optim.SGD(model.parameters(), lr = start_lr)

    lrs, losses = [], []
    smoothed_loss = None
    best_loss = float('inf')

    data_iter = iter(train_loader)

    for step in range(n_steps):
        try:
            bX, by = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            bX, by = next(data_iter)

        bX, by = bX.to(device), by.to(device)
        optimizer.zero_grad()
        output = model(bX)
        loss = criterion(output, by)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        if smoothed_loss is None:
            smoothed_loss = current_loss
        else:
            smoothed_loss = smoothing * current_loss + (1 - smoothing) * smoothed_loss

        lrs.append(optimizer.param_groups[0]['lr'])
        losses.append(smoothed_loss)

        if step > 5 and smoothed_loss > 4 * best_loss:
            print(f'LR test: Loss diverged at LR = {lrs[-1]:.2e}. Stopping.')
            break

        best_loss = min(best_loss, smoothed_loss)

        for pg in optimizer.param_groups:
            pg['lr'] *= lr_multiplier

    if len(losses) > 5:
        grad = np.gradient(losses)
        best_idx = np.argmin(grad[5:]) + 5
        suggested_lr = lrs[best_idx]
        print(f'\nSuggested LR: {suggested_lr:.2e} (steepest loss decrease at step) {best_idx}')

    return lrs, losses

import copy
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

X = torch.randn(800, 20)
y = (X[:, :3].sum(1) > 0).float().unsqueeze(1)
loader = DataLoader(TensorDataset(X, y), batch_size = 32, shuffle = True)

model_for_test = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), 
                               nn.Linear(32, 1)).to(device)
criterion = nn.BCEWithLogitsLoss()

model_copy = copy.deepcopy(model_for_test)
lrs, losses = lr_range_test(model_copy, loader, criterion, device, start_lr = 1e-6, end_lr = 1.0, n_steps = 80)

print(f"\n{'LR':>12}  {'Loss':>10}")
step_interval = max(1, len(lrs) // 15)
for i in range(0, len(lrs), step_interval):
    bar = "█" * min(30, int(losses[i] * 30 / (max(losses) + 1e-8)))
    print(f"  {lrs[i]:>10.2e}  {losses[i]:>8.4f}  {bar}")


Suggested LR: 5.01e+05 (steepest loss decrease at step) 36

          LR        Loss
    1.00e+03    0.6926  █████████████████████████████
    2.37e+03    0.6911  █████████████████████████████
    5.62e+03    0.6919  █████████████████████████████
    1.33e+04    0.6934  █████████████████████████████
    3.16e+04    0.6938  █████████████████████████████
    7.50e+04    0.6933  █████████████████████████████
    1.78e+05    0.6933  █████████████████████████████
    4.22e+05    0.6941  █████████████████████████████
    1.00e+06    0.6931  █████████████████████████████
    2.37e+06    0.6938  █████████████████████████████
    5.62e+06    0.6939  █████████████████████████████
    1.33e+07    0.6936  █████████████████████████████
    3.16e+07    0.6941  █████████████████████████████
    7.50e+07    0.6941  █████████████████████████████
    1.78e+08    0.6943  █████████████████████████████
    4.22e+08    0.6934  █████████████████████████████


In [9]:
# LR Schedules

# 1. StepLR
from torch.optim.lr_scheduler import StepLR
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = StepLR(optimizer, step_size = 10, gamma = 0.5)

for epoch in range(40):
    train_one_epoch(model, optimizer, ...)
    scheduler.step()
    print(f'Epoch {epoch}: LR = {scheduler.get_last_lr()[0]:.6f}')

# 2. MultiStepLR
from torch.optim.lr_scheduler import MultiStepLR
scheduler = MultiStepLR(optimizer, milestones = [30, 60, 90], gamma = 0.1)

# 3. CosineAnnealingLR
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max = 50, eta_min = 1e-6)
for epoch in range(50):
    train_one_epoch(...)
    scheduler.step()

# 4. ReduceLROnPlateau
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 5, min_lr = 1e-7, verbose = True)
for epoch in range(100):
    train_loss = train_one_epoch(...)
    val_loss = validate(...)
    
    scheduler.step(val_loss)

# 5. CosineAnnealingWarmRestarts
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0 = 10, T_mult = 2, eta_min = 1e-6)


NameError: name 'train_one_epoch' is not defined

In [14]:
# LR Strategies

# 1. Warmup Strategy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR

def build_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs, target_lr, min_lr = 1e-7):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)
        else:
            progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
            cosine = 0.5 * (1.0 + torch.cos(torch.tensor(3.14159 * progress)).item())
            min_scale = min_lr / target_lr
            return min_scale + (1.0 - min_scale) * cosine

    return LambdaLR(optimizer, lr_lambda = lr_lambda)

model = nn.Linear(10, 1)
optimizer = optim.AdamW(model.parameters(), lr = 0.001)

scheduler = build_warmup_cosine_scheduler(optimizer, warmup_epochs = 5, total_epochs = 50, 
                                         target_lr = 0.001, min_lr = 1e-6)
print(f"{'Epoch':>6} | {'LR':>12}")
for epoch in range(50):
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    if epoch < 8 or epoch % 10 == 0 or epoch >= 45:
        bar = "█" * int(lr / 0.001 * 20)
        print(f"{epoch+1:>6} | {lr:>12.6f}  {bar}")

 Epoch |           LR
     1 |     0.000200  ████
     2 |     0.000400  ████████
     3 |     0.000600  ████████████
     4 |     0.000800  ████████████████
     5 |     0.001000  ████████████████████
     6 |     0.000999  ███████████████████
     7 |     0.000995  ███████████████████
     8 |     0.000989  ███████████████████
    11 |     0.000957  ███████████████████
    21 |     0.000719  ██████████████
    31 |     0.000380  ███████
    41 |     0.000096  █
    46 |     0.000020  
    47 |     0.000012  
    48 |     0.000006  
    49 |     0.000002  
    50 |     0.000001  


C:\Users\bisht\AppData\Local\Temp\ipykernel_5928\2864363973.py:28: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


In [12]:
# Correctly Calling Schedulers
# 1. Epoch-based schedulers for StepLR, MultiStepLR, CosineAnnealingWarmRestarts
# Call scheduler.step() Once per epoch, after the training loop
for epoch in range(n_epochs):
    model.train()
    for bX, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bX), by)
        loss.backward()
        optimizer.step()
    scheduler.step()

# 2. ReduceLROnPlateau -- Needs a metrics - call after computing validation loss
for epoch in range(n_epochs):
    train_loss = train_one_epoch(...)
    val_loss = validate(...)
    scheduler.step(val_loss)

# 3. Step-based schedulers (per batch, not per epoch) --- CosineAnnealingWarmRestarts can be called 
# per step for finer control
for epoch in range(n_epochs):
    for batch_idx, (bX, by) in enumerate(train_loader):
        optimizer.zero_grad()
        loss = criterion(model(bX), by)
        loss.backward()
        optimizer.step()
        scheduler.step(epoch + batch_idxx / len(train_loader))

NameError: name 'n_epochs' is not defined

In [19]:
# Complete Training Loop with Scheduler
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

n = 2000
X = torch.randn(n, 20)
y = (X[:, :3].sum(1) > 0).long()   # multiclass: 2 classes

X_tr, X_val = X[:1600], X[1600:]
y_tr, y_val = y[:1600], y[1600:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=128, shuffle=False)

# ─── Model ───
model = nn.Sequential(
    nn.Linear(20, 128), nn.ReLU(), nn.BatchNorm1d(128),
    nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64),
    nn.Linear(64, 2)).to(device)

n_epochs = 60
warmup_epochs = 5
optimizer = optim.AdamW(model.parameters(), lr = 1e-3, weight_decay = 0.01)
warm_scheduler = LambdaLR(optimizer, lr_lambda = lambda epoch: min(1.0, (epoch + 1) / warmup_epochs))
cosine_scheduler = CosineAnnealingLR(optimizer, T_max = n_epochs - warmup_epochs, eta_min = 1e-6)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
history = []

for epoch in range(n_epochs):
    model.train()
    train_loss, correct, total = 0.0, 0, 0

    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bX)
        loss = criterion(logits, by)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
        optimizer.step()

        train_loss += loss.item() * bX.size(0)
        correct += (logits.argmax(1) == by).sum().item()
        total += bX.size(0)

    if epoch < warmup_epochs:
        warm_scheduler.step()
    else:
        cosine_scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for bX, by in val_loader:
            bX, by = bX.to(device), by.to(device)
            logits = model(bX)
            loss = criterion(logits, by)
            val_loss += loss.item() * bX.size(0)
            val_correct += (logits.argmax(1) == by).sum().item()
            val_total += bX.size(0)

    t_loss = train_loss / total
    v_loss = val_loss / val_total
    t_acc = correct / total * 100
    v_acc = val_correct / val_total * 100

    history.append({'epoch' : epoch + 1, 'train_loss' : t_loss, 'val_loss' : v_loss, 'train_acc' : t_acc,
                   'val_acc' : v_acc, 'lr' : current_lr})
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), 'best_model.pth')

    if (epoch + 1) % 10 == 0 or epoch < 6:
        print(f"Epoch {epoch+1:>3} | LR: {current_lr:.2e} | "
              f"Train: {t_loss:.4f} ({t_acc:.1f}%) | Val: {v_loss:.4f} ({v_acc:.1f}%)")

print(f"\nBest validation accuracy: {best_val_acc:.1f}%")

print("\nLR schedule across epochs:")
for record in history[::5]:
    bar = "█" * int(record['lr'] / 1e-3 * 20)
    print(f"  Epoch {record['epoch']:>3}: LR={record['lr']:.2e}  {bar}")

Epoch   1 | LR: 4.00e-04 | Train: 0.6122 (66.1%) | Val: 0.5498 (76.2%)
Epoch   2 | LR: 6.00e-04 | Train: 0.3580 (86.3%) | Val: 0.2987 (88.5%)
Epoch   3 | LR: 8.00e-04 | Train: 0.2204 (92.6%) | Val: 0.2001 (92.2%)
Epoch   4 | LR: 1.00e-03 | Train: 0.1508 (95.1%) | Val: 0.1569 (95.2%)
Epoch   5 | LR: 1.00e-03 | Train: 0.1144 (96.2%) | Val: 0.1363 (94.0%)
Epoch   6 | LR: 9.99e-04 | Train: 0.0876 (97.7%) | Val: 0.1242 (94.8%)
Epoch  10 | LR: 9.80e-04 | Train: 0.0567 (98.4%) | Val: 0.1207 (94.8%)
Epoch  20 | LR: 8.28e-04 | Train: 0.0290 (98.8%) | Val: 0.1377 (94.2%)
Epoch  30 | LR: 5.72e-04 | Train: 0.0273 (99.0%) | Val: 0.1339 (94.8%)
Epoch  40 | LR: 2.93e-04 | Train: 0.0048 (100.0%) | Val: 0.1430 (94.2%)
Epoch  50 | LR: 8.03e-05 | Train: 0.0054 (100.0%) | Val: 0.1625 (93.0%)
Epoch  60 | LR: 1.00e-06 | Train: 0.0100 (99.8%) | Val: 0.1566 (93.5%)

Best validation accuracy: 96.5%

LR schedule across epochs:
  Epoch   1: LR=4.00e-04  ████████
  Epoch   6: LR=9.99e-04  ███████████████████
  Ep

In [20]:
# Monitoring Optimizers
def monitor_training_health(model, optimizer, loss, step):
    if step % 50 != 0:
        return

    current_lor = optimizer.param_groups[0]['lr']
    grad_norms = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norms[name] = param.grad.norm().item()

    max_grad = max(grad_norms.values()) if grad_norms else 0
    min_grad = min(grad_norms.values()) if grad_norms else 0

    weight_norms = {name : param.norm().item()
                   for name, param in model.named_parameters() if 'weight' in name}

    print(f"\n[Step {step}] LR={current_lr:.2e} | Loss={loss:.4f}")
    print(f"  Gradient norm: max={max_grad:.4f}, min={min_grad:.6f}")
    print(f"  Weight norm:   max={max(weight_norms.values()):.4f}, "
          f"min={min(weight_norms.values()):.4f}")

    if max_grad > 100:
        print("  ⚠ WARNING: Large gradients detected — consider gradient clipping!")
    if max_grad < 1e-6:
        print("  ⚠ WARNING: Vanishing gradients detected — check init/activation!")
    if torch.isnan(loss):
        print("  ❌ ERROR: NaN loss! Check data, LR, and loss function.")

In [23]:
# Mini Exercise: Training Pipeline to compare optimizer and scheduler combos
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import TensorDataset, DataLoader
import copy

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

n = 3000
X = torch.randn(n, 30)
y = (X[:, :4].sum(1) - X[:, 4:6].sum(1) > 0).long()

X_tr, X_val = X[: 2400], X[2400: ]
y_tr, y_val = y[: 2400], y[2400: ]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size = 64, shuffle = True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size = 128, shuffle = False)

def make_model():
    m = nn.Sequential(
        nn.Linear(30, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Linear(128, 64), nn.ReLU(),
        nn.BatchNorm1d(64), nn.Linear(64, 2)).to(device)
    for layer in m:
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity = 'relu')
            nn.init.zeros_(layer.bias)
    return m

def run_experiment(optimizer_fn, scheduler_fn, n_epochs = 60, desc = ''):
    model = make_model()
    optimizer = optimizer_fn(model)
    scheduler, sched_mode = scheduler_fn(optimizer)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    hist = []

    for epoch in range(n_epochs):
        model.train()
        correct, total, t_loss = 0, 0, 0.0
        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            out  = model(bX)
            loss = criterion(out, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss  += loss.item() * bX.size(0)
            correct += (out.argmax(1) == by).sum().item()
            total   += bX.size(0)

        # Validation
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for bX, by in val_loader:
                bX, by = bX.to(device), by.to(device)
                out    = model(bX)
                loss   = criterion(out, by)
                v_loss    += loss.item() * bX.size(0)
                v_correct += (out.argmax(1) == by).sum().item()
                v_total   += bX.size(0)

        v_acc = v_correct / v_total * 100
        best_val_acc = max(best_val_acc, v_acc)

        hist.append({
            'val_acc': v_acc,
            'val_loss': v_loss / v_total,
            'lr': optimizer.param_groups[0]['lr']
        })

        if sched_mode == 'plateau':
            scheduler.step(v_loss / v_total)
        elif sched_mode == 'epoch':
            scheduler.step()

    return best_val_acc, hist

n_epochs = 60
experiments = {
    'Adam, fixed LR' : (
        lambda m: optim.Adam(m.parameters(), lr = 1e-3),
        lambda opt: (StepLR(opt, step_size = 999), 'epoch')),
    'AdamW + StepLR' : (
        lambda m: optim.AdamW(m.parameters(), lr = 1e-3, weight_decay = 0.01),
        lambda opt: (StepLR(opt, step_size = 20, gamma = 0.1), 'epoch')),
    'AdamW + CosineAnnealing' : (
        lambda m: optim.AdamW(m.parameters(), lr = 1e-3, weight_decay = 0.01),
        lambda opt: (CosineAnnealingLR(opt, T_max = n_epochs, eta_min = 1e-6), 'epoch')),
    'AdamW + ReduceLROnPlateau' : (
        lambda m: optim.AdamW(m.parameters(), lr = 1e-3, weight_decay = 0.01),
        lambda opt: (ReduceLROnPlateau(opt, patience = 5, factor = 0.5, min_lr = 1e-7), 'plateau')),
    'SGD+Momentum + CosineAnnealing' : (
        lambda m: optim.SGD(m.parameters(), lr = 0.05, momentum = 0.9, nesterov = True, weight_decay = 1e-4),
        lambda opt: (CosineAnnealingLR(opt, T_max = n_epochs, eta_min = 1e-6), 'epoch'))
}

print(f"\n{'Configuration':<35} {'Best Val Acc':>13} {'LR at Epoch 60':>15}")
all_results = {}
for desc, (opt_fn, sched_fn) in experiments.items():
    best_acc, hist = run_experiment(opt_fn, sched_fn, n_epochs=n_epochs, desc=desc)
    final_lr = hist[-1]['lr']
    all_results[desc] = (best_acc, hist)
    print(f"{desc:<35} {best_acc:>12.2f}%  {final_lr:>14.2e}")

print("\n── Epochs to reach 80% validation accuracy ──")
for desc, (_, hist) in all_results.items():
    epoch_80 = next((i+1 for i, h in enumerate(hist) if h['val_acc'] >= 80), None)
    print(f"  {desc:<35}: {epoch_80 if epoch_80 else 'Never reached'}")


Configuration                        Best Val Acc  LR at Epoch 60
Adam, fixed LR                             95.33%        1.00e-03
AdamW + StepLR                             95.33%        1.00e-05
AdamW + CosineAnnealing                    95.50%        1.68e-06
AdamW + ReduceLROnPlateau                  95.33%        7.81e-06
SGD+Momentum + CosineAnnealing             95.00%        3.53e-05

── Epochs to reach 80% validation accuracy ──
  Adam, fixed LR                     : 1
  AdamW + StepLR                     : 1
  AdamW + CosineAnnealing            : 1
  AdamW + ReduceLROnPlateau          : 1
  SGD+Momentum + CosineAnnealing     : 1
